# EDA — PaySim: detección de fraude bancario

Análisis exploratorio del dataset sintético **PaySim** (`ealaxi/paysim1`), que simula un mes de transacciones de un servicio de dinero móvil, con fraude inyectado en la columna `isFraud`.

Objetivo de este notebook: entender la forma del desbalance de clases, qué tipos de transacción concentran el fraude, y qué señales (montos, discrepancias de saldo) separan mejor las clases — para justificar las decisiones tomadas en `src/features/build_features.py` y `src/models/train.py`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_raw_data
from src.data.preprocessing import clean_data
from src.features.build_features import build_features

# Paleta categórica validada (colorblind-safe) — misma usada en src/models/visualize.py
COLOR_NOFRAUD = "#2a78d6"
COLOR_FRAUD = "#eb6834"
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
GRIDLINE = "#e1e0d9"

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = GRIDLINE
plt.rcParams["axes.edgecolor"] = INK_SECONDARY


## 1. Carga y estructura del dataset


In [ ]:
df = load_raw_data()
print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
df.head()


In [ ]:
df.info()


In [ ]:
df.isna().sum().rename("valores_nulos")


## 2. Distribución de la clase objetivo (`isFraud`)

El fraude es un evento extremadamente raro dentro del dataset. Esta proporción es la razón por la que `src/models/train.py` usa `class_weight="balanced"` / `scale_pos_weight` y por la que la comparación de modelos se hace con **PR-AUC** en vez de accuracy o ROC-AUC.


In [ ]:
fraud_counts = df["isFraud"].value_counts().sort_index()
fraud_pct = df["isFraud"].value_counts(normalize=True).sort_index() * 100

for label, (count, pct) in enumerate(zip(fraud_counts, fraud_pct)):
    tag = "Fraude" if label == 1 else "No fraude"
    print(f"{tag} ({label}): {count:,} transacciones ({pct:.4f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["No fraude", "Fraude"], fraud_counts.values, color=[COLOR_NOFRAUD, COLOR_FRAUD])
ax.set_yscale("log")
ax.set_ylabel("Número de transacciones (escala log)", color=INK_SECONDARY)
ax.set_title("Distribución de isFraud (escala logarítmica)", color=INK_PRIMARY)
plt.show()


## 3. Fraude por tipo de transacción

PaySim solo inyecta fraude en transferencias de dinero entre cuentas, no en pagos o depósitos. Confirmarlo aquí valida por qué `type` (codificado como dummies en `preprocessing.py`) es una feature tan informativa.


In [ ]:
type_fraud = pd.crosstab(df["type"], df["isFraud"])
type_fraud.columns = ["No fraude", "Fraude"]
type_fraud["pct_fraude"] = (type_fraud["Fraude"] / (type_fraud["Fraude"] + type_fraud["No fraude"])) * 100
type_fraud.sort_values("pct_fraude", ascending=False)


In [ ]:
fraud_by_type = df[df["isFraud"] == 1]["type"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(fraud_by_type.index, fraud_by_type.values, color=COLOR_FRAUD)
ax.set_ylabel("Transacciones fraudulentas", color=INK_SECONDARY)
ax.set_title("Fraude por tipo de transacción", color=INK_PRIMARY)
plt.show()


## 4. Distribución de montos (`amount`)

Comparamos la distribución de montos entre transacciones fraudulentas y legítimas, restringido a los tipos donde ocurre fraude (`TRANSFER`, `CASH_OUT`).


In [ ]:
relevant = df[df["type"].isin(["TRANSFER", "CASH_OUT"])]

print("Estadísticas de 'amount' — No fraude:")
print(relevant.loc[relevant["isFraud"] == 0, "amount"].describe())
print("\nEstadísticas de 'amount' — Fraude:")
print(relevant.loc[relevant["isFraud"] == 1, "amount"].describe())


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bins = np.logspace(0, np.log10(relevant["amount"].max() + 1), 50)

ax.hist(relevant.loc[relevant["isFraud"] == 0, "amount"], bins=bins, alpha=0.6, label="No fraude", color=COLOR_NOFRAUD, density=True)
ax.hist(relevant.loc[relevant["isFraud"] == 1, "amount"], bins=bins, alpha=0.6, label="Fraude", color=COLOR_FRAUD, density=True)
ax.set_xscale("log")
ax.set_xlabel("amount (escala log)", color=INK_SECONDARY)
ax.set_ylabel("Densidad", color=INK_SECONDARY)
ax.set_title("Distribución de montos: fraude vs. no fraude (TRANSFER/CASH_OUT)", color=INK_PRIMARY)
ax.legend(frameon=False)
plt.show()


## 5. Discrepancias de saldo (features derivadas)

`build_features.py` calcula `errorBalanceOrig` y `errorBalanceDest`: la diferencia entre el saldo esperado tras la transacción y el saldo real reportado. En transacciones legítimas esta discrepancia debería ser ~0; valores distintos de 0 son una señal fuerte de fraude (p. ej. cuentas destino que no reciben el dinero que deberían).


In [ ]:
df_feat = build_features(df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, col, title in zip(
    axes,
    ["errorBalanceOrig", "errorBalanceDest"],
    ["Discrepancia de saldo — origen", "Discrepancia de saldo — destino"],
):
    ax.boxplot(
        [df_feat.loc[df_feat["isFraud"] == 0, col], df_feat.loc[df_feat["isFraud"] == 1, col]],
        tick_labels=["No fraude", "Fraude"],
        showfliers=False,
        patch_artist=True,
        boxprops=dict(facecolor=COLOR_NOFRAUD, alpha=0.5),
    )
    ax.set_title(title, color=INK_PRIMARY)
    ax.set_ylabel(col, color=INK_SECONDARY)

fig.tight_layout()
plt.show()


## 6. Patrón temporal (`step`)

`step` representa horas simuladas (1 mes ≈ 744 steps). Revisamos si el fraude se concentra en ciertos momentos del periodo simulado.


In [ ]:
fraud_by_step = df[df["isFraud"] == 1].groupby("step").size()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(fraud_by_step.index, fraud_by_step.values, color=COLOR_FRAUD, linewidth=1.5)
ax.set_xlabel("step (hora simulada)", color=INK_SECONDARY)
ax.set_ylabel("Transacciones fraudulentas", color=INK_SECONDARY)
ax.set_title("Fraude a lo largo del tiempo simulado", color=INK_PRIMARY)
plt.show()


## 7. Correlación de features con `isFraud`

Aplicamos el pipeline completo de limpieza + features (`clean_data` + `build_features`) y observamos qué variables numéricas correlacionan más con la clase objetivo.


In [ ]:
df_model = clean_data(df)
df_model = build_features(df_model)

corr_with_target = df_model.corr(numeric_only=True)["isFraud"].drop("isFraud").sort_values()

fig, ax = plt.subplots(figsize=(7, 6))
colors = [COLOR_FRAUD if v < 0 else COLOR_NOFRAUD for v in corr_with_target.values]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors)
ax.axvline(0, color=INK_SECONDARY, linewidth=1)
ax.set_xlabel("Correlación de Pearson con isFraud", color=INK_SECONDARY)
ax.set_title("Correlación de features con la clase objetivo", color=INK_PRIMARY)
fig.tight_layout()
plt.show()


## 8. Conclusiones para el modelado

- El desbalance de clases es extremo (fraude ≈ 0.1% del dataset) → usar `class_weight="balanced"` / `scale_pos_weight` y evaluar con **PR-AUC**, no accuracy.
- El fraude solo aparece en `TRANSFER` y `CASH_OUT` → el one-hot encoding de `type` es una feature clave.
- Las discrepancias de saldo (`errorBalanceOrig`, `errorBalanceDest`) separan bien las clases → justifican la ingeniería de features en `src/features/build_features.py`.
- Con estas señales, el siguiente paso es entrenar y comparar modelos: ver `src/models/train.py` (`python -m src.models.train`), que entrena Regresión Logística, Random Forest y XGBoost, y selecciona el mejor por PR-AUC.
